In [11]:
import torch
import pandas as pd
import os
import sys
sys.path.append('../')

In [12]:
def count_parameters(path: str):
    model: dict = torch.load(os.path.join(path, "model.pth"), map_location='cpu', weights_only=True)
    num_params = 0
    num_params_non_zero = 0
    for k, v in model.items():
        num_params += v.numel()
        num_params_non_zero += (v != 0).sum().item()
    return int(num_params_non_zero)

In [13]:
models = [
    ("/local/scratch/clmn1/videoNCA/CataractsDataset/charmed-mountain-80", "NCA"),
    ("/local/scratch/clmn1/videoNCA/CataractsDataset/snowy-vortex-18", "SegFormer"),
    ("/local/scratch/clmn1/videoNCA/CataractsDataset/colorful-blaze-8", "UNet"),
    ("/local/scratch/clmn1/videoNCA/CataractsDataset/jumping-galaxy-22", "SwinUNetv2"),
]
df = []
for model in models:
    dices = 100 * pd.read_csv(os.path.join(model[0], "dices_val.csv"))
    maDice = dices.mean(axis=0).mean()
    maDice_std = dices.mean(axis=0).std()
    miDice = dices.stack().mean()
    miDice_std = dices.stack().std()
    dices = dices.mean()
    dices.name = model[1]
    dices["maDice"] = maDice
    dices["maDice std"] = maDice_std
    dices["miDice"] = miDice
    dices["miDice std"] = miDice_std
    dices["num params"] = count_parameters(model[0])
    df.append(dices)
df = pd.DataFrame(df)
df["num params"] = df["num params"].astype(int)
df

,Pupil (1),Surgical Tape (2),Hand (3),Eye Retractors (4),Iris (5),Skin (6),Cornea (7),Cannula (8),Cap. Cystotome (9),Tissue Forceps (10),...,Lens Injector (13),I/A Handpiece (14),Secondary Knife (15),Micromanipulator (16),Cap. Forceps (17),maDice,maDice std,miDice,miDice std,num params
NCA,93.792609,85.434175,78.167378,73.057880,86.252248,75.896936,92.307032,60.202500,74.303672,71.354218,...,76.121312,81.730461,79.540513,58.168812,25.446302,75.002071,15.948812,82.080158,17.972196,42505
SegFormer,94.709976,86.624185,73.460327,75.048076,87.690530,80.539836,93.715977,65.949355,33.482527,76.504274,...,65.146338,82.099200,78.182681,59.053559,11.663292,70.782868,21.243777,82.925912,20.543473,3719283
UNet,81.981054,67.422156,72.175046,53.117032,69.442293,55.478571,86.785789,37.396878,32.467563,24.452021,...,0.024146,57.871984,0.546490,41.681109,0.003265,43.075325,29.546224,63.710123,25.946286,68332580
SwinUNetv2,93.738728,83.801353,75.431706,72.311659,86.007395,68.582454,90.784131,57.007575,43.904790,71.522703,...,57.410105,81.080180,66.209972,52.934881,9.641779,67.427351,20.198945,79.001073,20.845000,27941700


In [14]:
df_latex = df[["maDice", "maDice std", "miDice", "miDice std", "num params"]].copy()
for metric in ["miDice", "maDice"]:
    mean_col = metric
    std_col = metric + " std"
    df_latex[metric] = (
                df_latex[mean_col].map("{:.1f}".format)
                + " $\\pm$ "
                + df_latex[std_col].map("{:.1f}".format)
            )
    df_latex = df_latex.drop(columns=[std_col])

In [15]:
df_latex

,maDice,miDice,num params
NCA,75.0 $\pm$ 15.9,82.1 $\pm$ 18.0,42505
SegFormer,70.8 $\pm$ 21.2,82.9 $\pm$ 20.5,3719283
UNet,43.1 $\pm$ 29.5,63.7 $\pm$ 25.9,68332580
SwinUNetv2,67.4 $\pm$ 20.2,79.0 $\pm$ 20.8,27941700


In [16]:
formatters =({
    "num params": lambda x: f"{int(x):,}"
})

df_latex = df_latex.to_latex(formatters=formatters)

print(df_latex)


\begin{tabular}{lllr}
\toprule
 & maDice & miDice & num params \\
\midrule
NCA & 75.0 $\pm$ 15.9 & 82.1 $\pm$ 18.0 & 42,505 \\
SegFormer & 70.8 $\pm$ 21.2 & 82.9 $\pm$ 20.5 & 3,719,283 \\
UNet & 43.1 $\pm$ 29.5 & 63.7 $\pm$ 25.9 & 68,332,580 \\
SwinUNetv2 & 67.4 $\pm$ 20.2 & 79.0 $\pm$ 20.8 & 27,941,700 \\
\bottomrule
\end{tabular}

